# Mad Eye Moody Lens — Detection Studio Cloud (Google Colab)

Run image detection in **your own Google Colab session** using the existing modular model plugins from the `ruturaj-dev` branch. This notebook is an add-on to the website; it does not change or replace the existing local application.

**What this notebook does**
1. Mount your own Google Drive and verify the two pretrained IAPL files.
2. Clone the existing Mad Eye Moody Lens repository and the official IAPL research source into temporary Colab storage.
3. Reuse the existing model registry, adapter and inference manager.
4. Copy model files to Colab's local disk, select a discovered model, upload an image and run inference.

**Before running:** push `detection-studio/` and this `detection-studio-cloud/` folder to the public `ruturaj-dev` branch. Upload your model files to Drive using the paths in `detection-studio-cloud/drive-template/README.md`.

This is an interactive notebook, **not** an always-on public API. Each visitor runs their own notebook session. Google Colab's GPU availability and session limits vary, and no GPU is guaranteed.

## 1. Configure the repository and model paths

If you rename your GitHub repository or branch, update the values below. The defaults match the current Mad Eye Moody Lens repository.

In [ ]:
from pathlib import Path

GITHUB_REPO = "https://github.com/Tanish-30-08-2006/Mad-Eye-Moody-Lens.git"
GITHUB_BRANCH = "ruturaj-dev"
PROJECT_DIR = Path("/content/Mad-Eye-Moody-Lens")
DETECTION_DIR = PROJECT_DIR / "detection-studio"

DRIVE_MODEL_ROOT = Path("/content/drive/MyDrive/MadEyeMoodyLens/DetectionStudioCloud/models")
DRIVE_IAPL = DRIVE_MODEL_ROOT / "iapl-sd14" / "checkpoint_best_acc_sd14.pth"
DRIVE_CLIP = DRIVE_MODEL_ROOT / "shared" / "clip" / "ViT-L-14.pt"
DEFAULT_MODEL_ID = "iapl-sd14"

print("Repository:", GITHUB_REPO)
print("Branch:", GITHUB_BRANCH)
print("Detection Studio:", DETECTION_DIR)
print("Drive model directory:", DRIVE_MODEL_ROOT)

## 2. Connect Google Drive and verify model files

Run this cell and approve access to your own Drive. The notebook only reads the two specified model files and copies them into the temporary Colab runtime.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

missing = [str(path) for path in (DRIVE_IAPL, DRIVE_CLIP) if not path.is_file()]
if missing:
    raise FileNotFoundError(
        "Missing model file(s):\n" + "\n".join(missing) +
        "\nCheck detection-studio-cloud/drive-template/README.md for the required Drive layout."
    )

for path in (DRIVE_IAPL, DRIVE_CLIP):
    print(f"Found {path.name}: {path.stat().st_size / (1024**3):.2f} GiB")

## 3. Clone the existing project and official IAPL source

The repository checkout supplies the existing Detection Studio plugin system and adapter. The official IAPL implementation is cloned separately because it is upstream research source and is intentionally not vendored into your application repository.

In [ ]:
import subprocess

if not (DETECTION_DIR / "backend" / "app" / "plugins.py").is_file():
    if PROJECT_DIR.exists():
        raise RuntimeError(
            f"{PROJECT_DIR} exists but is not a valid project checkout. "
            "Delete only this temporary /content/Mad-Eye-Moody-Lens folder in Colab and rerun."
        )
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPO, str(PROJECT_DIR)],
        check=True,
    )

required_project_files = [
    DETECTION_DIR / "backend" / "app" / "plugins.py",
    DETECTION_DIR / "backend" / "app" / "manager.py",
    DETECTION_DIR / "models" / "iapl-sd14" / "model.yaml",
    DETECTION_DIR / "models" / "iapl-sd14" / "adapter.py",
]
missing_project = [str(path) for path in required_project_files if not path.is_file()]
if missing_project:
    raise FileNotFoundError(
        "The public branch is missing required Detection Studio files:\n" + "\n".join(missing_project) +
        "\nPush your current detection-studio/ folder to the ruturaj-dev branch first."
    )

IAPL_REPO = DETECTION_DIR / "third_party" / "IAPL"
IAPL_REPO.parent.mkdir(parents=True, exist_ok=True)
if not (IAPL_REPO / "main.py").is_file():
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/liyih/IAPL.git", str(IAPL_REPO)],
        check=True,
    )

print("Detection Studio plugin system found.")
print("Official IAPL source:", IAPL_REPO)

## 4. Install IAPL dependencies without replacing Colab's CUDA PyTorch

The local project uses PyTorch 2.6 CPU. This notebook deliberately does **not** install that local CPU build. It keeps the PyTorch/torchvision build supplied by the active Colab runtime and installs only the IAPL plugin's non-PyTorch requirements.

After installation, check the detected versions and GPU. If a package installation requests a runtime restart, restart the runtime and rerun cells from the top.

In [ ]:
import sys
import subprocess

requirements = DETECTION_DIR / "models" / "iapl-sd14" / "requirements.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "PyYAML>=6,<7", "Pillow>=10,<13", "-r", str(requirements)],
    check=True,
)

import torch, torchvision
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU was assigned. CPU inference may be very slow.")

## 5. Copy model artifacts to Colab's local disk

Google Drive is persistent storage; `/content` is temporary local runtime storage. The model manager reads local files during inference, not the mounted Drive copies.

In [ ]:
import shutil

IAPL_PLUGIN_DIR = DETECTION_DIR / "models" / "iapl-sd14"
IAPL_WEIGHTS_DIR = IAPL_PLUGIN_DIR / "weights"
CLIP_DIR = DETECTION_DIR / "models" / "_shared" / "clip"
IAPL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
CLIP_DIR.mkdir(parents=True, exist_ok=True)

local_iapl = IAPL_WEIGHTS_DIR / "checkpoint_best_acc_sd14.pth"
local_clip = CLIP_DIR / "ViT-L-14.pt"
shutil.copy2(DRIVE_IAPL, local_iapl)
shutil.copy2(DRIVE_CLIP, local_clip)

print("IAPL checkpoint copied:", local_iapl, f"({local_iapl.stat().st_size / (1024**3):.2f} GiB)")
print("CLIP weights copied:", local_clip, f"({local_clip.stat().st_size / (1024**3):.2f} GiB)")

## 6. Discover available model plugins

The same plugin registry and inference manager used by the local application are imported from the existing `detection-studio/backend` package. Add a new model folder with its own `model.yaml` and adapter to the local project, push it to `ruturaj-dev`, and it will be discoverable on the next fresh Colab checkout (provided its artifacts and dependencies are also configured).

In [ ]:
import sys
sys.path.insert(0, str(DETECTION_DIR / "backend"))

from app.plugins import PluginRegistry
from app.manager import InferenceManager

registry = PluginRegistry(DETECTION_DIR / "models")
manager = InferenceManager(registry)
plugins = registry.list()

for plugin in plugins:
    info = plugin.public()
    print(f"{info['id']:<24} {info['status']:<14} {info['name']}")

ready_plugins = [plugin for plugin in plugins if plugin.selectable]
if not ready_plugins:
    raise RuntimeError("No model is ready. Verify model.yaml, adapter.py and required model files.")
print(f"\n{len(ready_plugins)} model(s) ready for inference.")

## 7. Select a detector

Only plugins marked ready by the registry are selectable. Models that are disabled, planned, or missing required artifacts are not offered for inference.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

options = [(f"{plugin.manifest.get('name', plugin.id)} — {plugin.id}", plugin.id) for plugin in ready_plugins]
model_selector = widgets.Dropdown(
    options=options,
    value=DEFAULT_MODEL_ID if any(plugin.id == DEFAULT_MODEL_ID for plugin in ready_plugins) else ready_plugins[0].id,
    description="Detector:",
    layout=widgets.Layout(width="85%"),
)
display(model_selector)

## 8. Upload an image and run detection

Upload one image when prompted. Inference runs in your own Colab session. IAPL uses test-time adaptation, so the first prediction includes model loading and adaptation costs.

In [ ]:
from google.colab import files
from io import BytesIO
from PIL import Image, ImageOps
from IPython.display import display

uploaded = files.upload()
if not uploaded:
    raise ValueError("No image uploaded.")

filename, content = next(iter(uploaded.items()))
image = ImageOps.exif_transpose(Image.open(BytesIO(content))).convert("RGB")
if min(image.size) < 32 or image.width * image.height > 40_000_000:
    raise ValueError("Unsupported image dimensions. Use at least 32x32 and at most 40 megapixels.")

preview_scale = min(1.0, 720 / image.width, 720 / image.height)
preview = image.resize((max(1, int(image.width * preview_scale)), max(1, int(image.height * preview_scale))))
display(preview)
print("File:", filename, "| dimensions:", image.size)
print("Selected detector:", model_selector.value)

result = manager.predict(model_selector.value, image)
print("\nPREDICTION:", result["predicted_label"])
print(f"Model score: {result['score'] * 100:.2f}%")
print("Score meaning:", result.get("score_semantics", "Model-specific score"))
print(f"Model load time: {result.get('model_load_time_ms', 0) / 1000:.2f} s")
print(f"Inference time: {result.get('inference_time_ms', 0) / 1000:.2f} s")
print("Device:", result.get("metadata", {}).get("device", "not reported"))
print("Metadata:", result.get("metadata", {}))
print("\nDisclaimer:", result.get("disclaimer", "Research-model output; not proof of authenticity."))

## 9. Run another image or switch models

- To run another image with the same model, rerun the upload/inference cell.
- To switch models, change the **Detector** dropdown and rerun the upload/inference cell.
- The existing `InferenceManager` reuses the active model and unloads it when switching to a different plugin.
- When finished, disconnect the Colab runtime to release temporary compute resources.

In [ ]:
# Optional: explicitly unload the active detector before ending the session.
# manager.unload()

## Notes, safety and limitations

- This notebook is an interactive Colab workflow, not a persistent public inference API or a website backend.
- Each visitor must open the notebook in their own Google account and run their own session. GPU access, usage limits, available hardware and session lifetime vary.
- The notebook clones source code from the public `ruturaj-dev` branch. Do not put credentials or private tokens in the notebook.
- `drive.mount()` grants notebook code access to the Google Drive account you authorize. Review the notebook code and only store the two model files in the specified folder if you prefer to limit what this workflow uses.
- Images are uploaded into the user's Colab session. This static website does not receive or store the images.
- IAPL uses test-time adaptation. Do not disable adaptation or change inference settings while claiming to reproduce the paper's reported results; treat altered configurations as separate experiments.
- The model's sigmoid score is model-specific and is not necessarily a calibrated probability. Do not treat a single result as proof of authenticity or manipulation.
- Review IAPL source and checkpoint licensing/terms before redistributing or using commercially.